In [12]:
# Celda 1: Imports y configuración


import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from statsmodels.regression.quantile_regression import QuantReg
from matplotlib.gridspec import GridSpec
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from src.config import get_birth_values
import importlib
import src.config as cfg
import matplotlib as mpl



np.set_printoptions(suppress=True)
pd.options.display.max_columns = None
pd.options.display.max_colwidth = None
pd.options.display.max_rows = None


In [13]:
# Celda 2: Cargar dataset
df = pd.read_csv('data/dataset_mrh_0.9.csv')
print(df.shape)
#df = df[df["Data_curator"] != "Otis 1954"]
print(df.shape)


(783, 11)
(783, 11)


In [14]:
# Celda 3: Preparar datos (unir DAF humano y filtrar roedores)
df_con_dummy = df.merge(
    df[df["Species"] == "Human"][["Parameter", "DAF"]],
    on="Parameter",
    suffixes=("", "_Human"),
)
df_con_dummy = df_con_dummy[df_con_dummy["Species"].isin(["Mouse", "Rat"])]

df_con_dummy.head()


,Species,Parameter,DAF,Reference,Obs,Quote,DOI,link,Cluster,Data_curator,GA(H+14),DAF_Human
0,Mouse,GE volume peak,15.75,"Charvet et al., 2017",NaN,"""The GE peak size occurs between the ages of ED 15.5 to ED 16 in mice, ED 80 in the macaque, and between GW 21 and 27 in humans. The growth trajectories from high-resolution diffusion MR scans also demonstrate that the peak in GE size occurs between GW 20 and GW 21 in humans (figure 5)""",https://doi.org/10.1098/rspb.2017.1169,NaN,Brain,Cottam 2024,NaN,154.00
1,Mouse,Cortical proliferative zone volume peak,15.00,"Charvet et al., 2017",NaN,"""The peak size of the CPP occurs between ED 14.5 and 15.5 in mice, on approximately ED 90 in macaques and on ED 182 in humans.""",https://doi.org/10.1098/rspb.2017.1169,NaN,Brain,Cottam 2024,NaN,168.00
4,Mouse,Corpus callosum peak volume,30.00,"Charvet et al., 2020",NaN,figure 4,https://doi.org/10.1093/cercor/bhz178,NaN,Brain,Cottam 2024,NaN,905.78
6,Rat,Auditory startle reaction,32.50,"Rybalko et al., 2015",NaN,"""First ASRs appeared on P10–P11 to intense low-frequency\r\ntones.""",https://doi.org/10.1016/j.bbr.2015.02.046,NaN,Brain,Cottam 2024,NaN,154.00
7,Mouse,Auditory startle reaction,31.00,"Fox, 1965",NaN,"""The development of the organs of special sense\r\nwas delayed until 10 to 12 days of age when the\r\nfirst auditory startle responses could be detected.""",https://doi.org/10.1016/0003-3472(65)90041-2,NaN,Brain,Cottam 2024,NaN,154.00


In [15]:
# Celda 5: Datos base para los ajustes entre humano y roedores
# Datos base para los ajustes entre humano y roedores
x = df_con_dummy["DAF_Human"].values
y = df_con_dummy["DAF"].values
D = (df_con_dummy["Species"] == "Rat").astype(int).values


In [16]:
# Celda 6: Definición de modelos y utilidades (wls_fit_joint)
# Model M1: fixed exponent (a = 0.25), shared
model_M1    = cfg.model_M1
model_M2    = cfg.model_M2
model_M3    = cfg.model_M3
model_M_log = cfg.model_M_log
model_M1_nc = cfg.model_M1_nc
model_M2_nc = cfg.model_M2_nc
model_M3_nc = cfg.model_M3_nc
colors_cfg  = cfg.colors


def wls_fit_joint(model_func, X, y, p0, maxfev=30000):
    """
    WLS fit for joint models where X = (x_human, D_species).
    Variance model: log(σ²) ~ c0 + c1·log(x_human)  (OLS on log-residuals).
    Returns (params_wls, wsrs, pcov, (c0, c1)) or None on convergence failure.
    """
    x_base = X[0]
    try:
        p_ols, _ = curve_fit(model_func, X, y, p0=p0, maxfev=maxfev)
    except RuntimeError:
        return None
    y_ols  = model_func(X, *p_ols)
    resid2 = np.maximum((y - y_ols) ** 2, np.finfo(float).eps)
    log_x  = np.log(np.maximum(x_base, np.finfo(float).eps))
    Z      = sm.add_constant(log_x)
    var_fit = sm.OLS(np.log(resid2), Z).fit()
    c0, c1  = var_fit.params[0], var_fit.params[1]
    sigma2_hat = np.maximum(np.exp(var_fit.predict(Z)), np.finfo(float).eps)
    sigma_hat  = np.sqrt(sigma2_hat)
    try:
        p_wls, pcov = curve_fit(model_func, X, y, p0=p_ols,
                                sigma=sigma_hat, absolute_sigma=False,
                                maxfev=maxfev)
    except RuntimeError:
        return None
    y_wls   = model_func(X, *p_wls)
    weights = 1.0 / sigma2_hat
    wsrs    = float(np.sum(weights * (y - y_wls) ** 2))
    return p_wls, wsrs, pcov, (c0, c1)




In [17]:


def comparar_modelos_conjuntos_loocv(x, y, D):
    """Compara modelos conjuntos usando WLS y LOOCV. 
    Devuelve: (tabla_métricas, diccionario_de_parámetros_y_pvalores)"""
    x, y, D = np.asarray(x, float), np.asarray(y, float), np.asarray(D, float)
    mask = (x > 0) & np.isfinite(x) & np.isfinite(y) & np.isfinite(D)
    x, y, D = x[mask], y[mask], D[mask]
    n_samples = len(y)
    
    print(f"⚠️ Ejecutando LOOCV para {n_samples} muestras. Esto puede tomar unos minutos...\n")

    keys_order = [ 'M8', 'M9', 'M10', 'M11', 'M12','M13','M14','M15','M16']

    model_configs = []
    for key in keys_order:
        entry = cfg.MODELS[key]
        func = getattr(cfg, entry['function'])
        p0 = list(entry.get('initial_params', []))
        label = entry.get('name', f"{key}")
        equation = entry.get('equation', '')
        eq_number = entry.get('equation_number', np.nan)
        model_configs.append((func, p0, label, equation, eq_number))

    indices = np.arange(n_samples)
    rows = []
    params_dict = {} 
    
    for func, p0, label, equation, eq_number in model_configs:
        fold_wrmse_list, n_failed = [], 0
        
        # --- LOOCV (Sin p-valores aquí) ---
        for i in range(n_samples):
            test_idx, train_idx = np.array([i]), np.setdiff1d(indices, [i])
            X_tr, y_tr = (x[train_idx], D[train_idx]), y[train_idx]
            X_te, y_te = (x[test_idx], D[test_idx]), y[test_idx]
            
            result = wls_fit_joint(func, X_tr, y_tr, p0)
            if result is None:
                n_failed += 1
                continue
                
            p_wls, _, _, (c0, c1) = result 
            y_pred = func(X_te, *p_wls)
            log_x_te = np.log(np.maximum(X_te[0], np.finfo(float).eps))
            w_te = 1.0 / np.maximum(np.exp(c0 + c1 * log_x_te), np.finfo(float).eps)
            fold_wrmse_list.append(np.sqrt(np.sum(w_te * (y_te - y_pred)**2) / np.sum(w_te)))
            
        # --- AJUSTE IN-SAMPLE (Aquí SÍ calculamos p-valores) ---
        
        result_full = wls_fit_joint(func, (x, D), y, p0)
        
        # Inicializar arrays vacíos por si falla
        params = np.full(len(p0), np.nan)
        p_values = np.full(len(p0), np.nan)
        se = np.full(len(p0), np.nan)  # <--- AÑADIR ESTO
        
        if result_full is not None:
            p_wls_full, wsrs, pcov, (c0_f, c1_f) = result_full 
            w_full = 1.0 / np.maximum(np.exp(c0_f + c1_f * np.log(np.maximum(x, np.finfo(float).eps))), np.finfo(float).eps)
            wrmse_in = np.sqrt(wsrs / np.sum(w_full))
            
            params = p_wls_full
            
            # --- CÁLCULO DE P-VALORES Y ERRORES ESTÁNDAR ---
            try:
                se = np.sqrt(np.diag(pcov)) # <--- AÑADIR ESTO
                t_stats = params / se
                dof = max(n_samples - len(p0), 1)
                p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=dof))
            except Exception:
                pass
        else:
            wrmse_in = np.inf
            
        # ... (líneas de la tabla limpia) ...

        # 2. Guardar coeficientes, p-valores y SE en el diccionario
        params_dict[label] = {
            'coefs': params,
            'p_values': p_values,
            'se': se  # <--- AÑADIR ESTO
        }
        # Inicializar arrays vacíos por si falla
        params = np.full(len(p0), np.nan)
        p_values = np.full(len(p0), np.nan)
        
        if result_full is not None:
            p_wls_full, wsrs, pcov, (c0_f, c1_f) = result_full 
            w_full = 1.0 / np.maximum(np.exp(c0_f + c1_f * np.log(np.maximum(x, np.finfo(float).eps))), np.finfo(float).eps)
            wrmse_in = np.sqrt(wsrs / np.sum(w_full))
            
            params = p_wls_full
            
            # --- CÁLCULO DE P-VALORES ---
            try:
                # 1. Error estándar de cada coeficiente (raíz de la diagonal de la matriz de covarianza)
                perr = np.sqrt(np.diag(pcov))
                # 2. Estadístico t (coeficiente / error estándar)
                t_stats = params / perr
                # 3. Grados de libertad (N muestras - k parámetros)
                dof = max(n_samples - len(p0), 1)
                # 4. P-valor de dos colas
                p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=dof))
            except Exception:
                # Si la matriz de covarianza tiene NaNs o no se puede calcular, dejamos el array de NaNs
                pass
        else:
            wrmse_in = np.inf
            
        # 1. Añadir a la tabla LIMPIA (sin parámetros ni p-valores)
        rows.append({
            'Model': label,
            'Equation': equation,
            'Equation number': int(eq_number) if not np.isnan(eq_number) else '',
            'k': len(p0),
            'In-sample WRMSE': wrmse_in,
            'CV-WRMSE': np.mean(fold_wrmse_list) if fold_wrmse_list else np.inf,
            'Fallas': n_failed
        })
                # 2. Guardar usando el KEY único (ej. 'M7'), no el nombre
        params_dict[key] = {
            'coefs': params,
            'p_values': p_values,
            'se': se
        }
        
    # --- CONSTRUIR TABLA ---
    tabla = pd.DataFrame(rows).sort_values(by=['CV-WRMSE', 'k']).reset_index(drop=True)
    min_wrmse = tabla['CV-WRMSE'].min()
        
    # --- FORMATEO PARA IMPRESIÓN EN CONSOLA ---
    tabla_print = tabla.copy()
    tabla_print['CV-WRMSE'] = tabla_print.apply(
        lambda r: f"**{r['CV-WRMSE']:.4f}**" if r['CV-WRMSE'] == min_wrmse else f"{r['CV-WRMSE']:.4f}", axis=1
    )
    tabla_print['In-sample WRMSE'] = tabla_print['In-sample WRMSE'].apply(
        lambda x: f"{x:.4f}" if np.isfinite(x) else "Fallo"
    )

    columnas_mostrar = ['Model', 'Equation number', 'Equation', 'k', 'In-sample WRMSE', 'CV-WRMSE', 'Fallas']
    
    print(f'\n{"="*90}')
    print(f' COMPARACIÓN DE MODELOS CONJUNTOS (Leave-One-Out CV)')
    print(f'{"="*90}')
    print(tabla_print[columnas_mostrar].to_string(index=False))
    print(f'{"="*90}\n')

    try:
        csv_path = 'results/tabla_loocv_dummy.csv'
        tabla.to_csv(csv_path, index=False, encoding='utf-8')
        print(f"✅ Tabla guardada en {csv_path}")
    except Exception as e:
        print(f"⚠️ No se pudo guardar CSV: {e}")

    return tabla, params_dict



In [18]:


# Ejecutar y desempaquetar la tupla
tabla_loocv, parametros_loocv = comparar_modelos_conjuntos_loocv(x, y, D)
csv_path = 'results/tabla_loocv_dummy.csv'
tabla_loocv.to_csv(csv_path, index=False, encoding='utf-8')
print(f"✅ Tabla de métricas guardada en {csv_path}")

⚠️ Ejecutando LOOCV para 421 muestras. Esto puede tomar unos minutos...


 COMPARACIÓN DE MODELOS CONJUNTOS (Leave-One-Out CV)
                  Model  Equation number                                Equation  k In-sample WRMSE   CV-WRMSE  Fallas
    Power (quarter) + c                9      $y = (b_0+b_1D)x^{1/4}+(c_0+c_1D)$  4          1.7826 **2.9706**       0
       Power (free) + c               11 $y = (b_0+b_1D)x^{a_0+a_1D}+(c_0+c_1D)$  6          1.6949     3.1207       0
            Logarithmic                8       $y = (b_0+b_1D)\ln(x)+(c_0+c_1D)$  4          2.0459     3.1777       0
     Power (shared) + c               10          $y = (b_0+b_1D)x^a+(c_0+c_1D)$  5          1.5885     3.2195       0
        Third-power + c               10      $y = (b_0+b_1D)x^{1/3}+(c_0+c_1D)$  4          2.1250     3.2347       0
Third-power no constant               11                 $y = (b_0+b_1D)x^{1/3}$  2          2.1959     3.2390       0
         Power (shared)               13

In [19]:
display(tabla_loocv)

,Model,Equation,Equation number,k,In-sample WRMSE,CV-WRMSE,Fallas
0,Power (quarter) + c,$y = (b_0+b_1D)x^{1/4}+(c_0+c_1D)$,9,4,1.782565,2.970573,0
1,Power (free) + c,$y = (b_0+b_1D)x^{a_0+a_1D}+(c_0+c_1D)$,11,6,1.694858,3.120652,0
2,Logarithmic,$y = (b_0+b_1D)\ln(x)+(c_0+c_1D)$,8,4,2.045868,3.177703,0
3,Power (shared) + c,$y = (b_0+b_1D)x^a+(c_0+c_1D)$,10,5,1.588461,3.219496,0
4,Third-power + c,$y = (b_0+b_1D)x^{1/3}+(c_0+c_1D)$,10,4,2.125012,3.234656,0
5,Third-power no constant,$y = (b_0+b_1D)x^{1/3}$,11,2,2.195858,3.238972,0
6,Power (shared),$y = (b_0+b_1D)x^a$,13,3,2.557874,3.396508,0
7,Power (free),$y = (b_0+b_1D)x^{a_0+a_1D}$,14,4,2.538356,3.444726,0
8,Power (quarter),$y = (b_0+b_1D)x^{1/4}$,12,2,3.966400,3.899505,0


In [20]:
display(parametros_loocv)


{'Logarithmic': {'coefs': array([ 4.77444683,  2.79105578, -5.16758109, -8.51840321]),
  'p_values': array([0., 0., 0., 0.]),
  'se': array([0.19080902, 0.30567434, 0.6229439 , 1.07893039])},
 'M16': {'coefs': array([ 0.39277581, -0.01146672,  2.83639707,  0.56056427]),
  'p_values': array([0.        , 0.48542898, 0.        , 0.00942738]),
  'se': array([0.01325566, 0.0164228 , 0.14765078, 0.21491937])},
 'Power (quarter) + c': {'coefs': array([ 8.72603862,  2.11183378, -9.56089822, -3.98776333]),
  'p_values': array([0.        , 0.00000172, 0.        , 0.00015769]),
  'se': array([0.29824524, 0.43510455, 0.67374836, 1.04568975])},
 'Power (shared) + c': {'coefs': array([ 0.31153744,  5.70947914,  1.00409343, -5.67406851, -1.92686187]),
  'p_values': array([0.        , 0.00000172, 0.0437295 , 0.00044958, 0.10542803]),
  'se': array([0.03205354, 1.17648608, 0.49637963, 1.60401852, 1.18749977])},
 'Power (free) + c': {'coefs': array([ 0.30948068, -0.03139606,  5.79126767,  2.85901318, -5

In [21]:
# Celda 9: Construir tabla de parámetros ANCHA para el reporte (Filtrado estricto)

def build_params_table_wide_only(parametros_loocv, models_config=cfg.MODELS):
    """Genera tabla Ancha solo para los modelos conjuntos deseados."""
    
    # --- FILTRO EXPLÍCITO PARA EVITAR MODELOS NO DESEADOS ---
    allowed_models = [ 'M8', 'M9', 'M10', 'M11', 'M12','M13','M14','M15','M16']
    
    def get_sig(pval):
        if pd.isna(pval): return ''
        return '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else '.' if pval < 0.1 else 'ns'

    rows_wide = []
    
    for model_key, model_cfg in models_config.items():
        # SI NO ESTÁ EN LA LISTA, LO SALTAMOS INMEDIATAMENTE
        if model_key not in allowed_models:
            continue
            
        label = model_cfg.get('name', model_key)
        equation = model_cfg.get('equation', '')
        p_names = model_cfg.get('param_names', [])
        
        # Leer del diccionario
        model_data = parametros_loocv.get(label)
        if model_data is None: continue
            
        coefs = model_data.get('coefs', [])
        p_vals = model_data.get('p_values', [])
        if len(coefs) == 0: continue
            
        # Construir la fila
        row_wide = {'Model': model_key, 'Name': label, 'Equation': equation}
        for i, val in enumerate(coefs):
            pname = p_names[i] if i < len(p_names) else f'p{i}'
            pval = p_vals[i] if i < len(p_vals) else np.nan
            row_wide[pname] = f"{round(float(val), 4)} {get_sig(pval)}" 
            
        rows_wide.append(row_wide)

    # Crear DataFrame y ordenar
    df_wide = pd.DataFrame(rows_wide)
    if not df_wide.empty:
        df_wide['_sort'] = df_wide['Model'].str.extract(r'M(\d+)').astype(int)
        df_wide = df_wide.sort_values('_sort').drop('_sort', axis=1).reset_index(drop=True)

    return df_wide

# ============================================================
# EJECUTAR Y MOSTRAR
# ============================================================
df_wide_reporte = build_params_table_wide_only(parametros_loocv)

print("TABLA DE COEFICIENTES PARA REPORTE")
print("(Significancia: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 'ns')\n")
display(df_wide_reporte)

try:
    csv_path = 'results/coef_p_dummy.csv'
    df_wide_reporte.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"\n✅ Tabla guardada en {csv_path}")
except Exception as e:
    print(f"\n⚠️ Error al guardar: {e}")

TABLA DE COEFICIENTES PARA REPORTE
(Significancia: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 'ns')



,Model,Name,Equation,b₀,b₁,c₀,c₁,a₀,a₁
0,M8,Logarithmic,$y = (b_0+b_1D)\ln(x)+(c_0+c_1D)$,4.7744 ***,2.7911 ***,-5.1676 ***,-8.5184 ***,NaN,NaN
1,M9,Power (quarter) + c,$y = (b_0+b_1D)x^{1/4}+(c_0+c_1D)$,8.726 ***,2.1118 ***,-9.5609 ***,-3.9878 ***,NaN,NaN
2,M10,Power (shared) + c,$y = (b_0+b_1D)x^a+(c_0+c_1D)$,5.7095 ***,1.0041 *,-5.6741 ***,-1.9269 ns,0.3115 ***,NaN
3,M11,Power (free) + c,$y = (b_0+b_1D)x^{a_0+a_1D}+(c_0+c_1D)$,5.7913 ***,2.859 ns,-5.7909 **,-4.6869 ns,0.3095 ***,-0.0314 ns
4,M12,Third-power + c,$y = (b_0+b_1D)x^{1/3}+(c_0+c_1D)$,4.8655 ***,0.6596 **,-4.3485 ***,-0.9813 ns,NaN,NaN
5,M13,Third-power no constant,$y = (b_0+b_1D)x^{1/3}$,3.4582 ***,0.6154 ***,NaN,NaN,NaN,NaN
6,M14,Power (quarter),$y = (b_0+b_1D)x^{1/4}$,4.9292 ***,1.3783 ***,NaN,NaN,NaN,NaN
7,M15,Power (shared),$y = (b_0+b_1D)x^a$,2.9305 ***,0.4226 ***,NaN,NaN,0.3841 ***,NaN
8,M16,Power (free),$y = (b_0+b_1D)x^{a_0+a_1D}$,2.8364 ***,0.5606 **,NaN,NaN,0.3928 ***,-0.0115 ns



✅ Tabla guardada en results/coef_p_dummy.csv


In [22]:
# Celda 10: Test de diferencias entre grupos (Significancia de Dummies)
# Ya no recalcula nada, solo extrae y formatea del diccionario.

def build_dummy_significance_table(parametros_loocv, models_config=cfg.MODELS):
    """
    Extrae solo los parámetros dummy (diferencia Rat - Mouse) y sus estadísticos.
    Requiere que cfg.MODELS tenga la clave 'dummy_param_indices' (ej: [1, 3]).
    """
    rows = []
    
    for model_key, model_cfg in models_config.items():
        label = model_cfg.get('name', model_key)
        p_names = model_cfg.get('param_names', [])
        
        # Leer índices de los parámetros dummy desde la configuración
        didx = model_cfg.get('dummy_param_indices', model_cfg.get('dummy_indices', []))
        if not didx: continue
            
        model_data = parametros_loocv.get(label)
        if model_data is None: continue
            
        coefs = model_data.get('coefs', [])
        se = model_data.get('se', [])
        p_vals = model_data.get('p_values', [])
        
        for i in didx:
            if i >= len(coefs): continue
            
            pname = p_names[i] if i < len(p_names) else f'p{i}'
            estimate = float(coefs[i])
            se_val = float(se[i]) if i < len(se) else np.nan
            pval = float(p_vals[i]) if i < len(p_vals) else np.nan
            t_stat = estimate / se_val if se_val > 0 else np.nan
            
            sig = ('***' if pval < 0.001 else '**' if pval < 0.01 else 
                   '*' if pval < 0.05 else '.' if pval < 0.1 else 'ns')
                   
            rows.append({
                'ModelKey': model_key, 'Model': label,
                'Param (Δ Rat-Mouse)': pname,
                'Estimate': round(estimate, 5), 'SE': round(se_val, 5),
                't': round(t_stat, 3), 'p-value': round(pval, 4), 'Sig': sig
            })
            
    df_sig = pd.DataFrame(rows)
    print("\n=== Significancia de parámetros de diferencia (Mouse vs Rat) ===")
    print("Sig codes:  0 '***'  0.001 '**'  0.01 '*'  0.05 '.'  0.1 'ns'\n")
    display(df_sig)

   

# Ejecutar
df_significance = build_dummy_significance_table(parametros_loocv)


=== Significancia de parámetros de diferencia (Mouse vs Rat) ===
Sig codes:  0 '***'  0.001 '**'  0.01 '*'  0.05 '.'  0.1 'ns'



,ModelKey,Model,Param (Δ Rat-Mouse),Estimate,SE,t,p-value,Sig
0,M8,Logarithmic,b₁,2.79106,0.30567,9.131,0.0000,***
1,M8,Logarithmic,c₁,-8.51840,1.07893,-7.895,0.0000,***
2,M9,Power (quarter) + c,b₁,2.11183,0.43510,4.854,0.0000,***
3,M9,Power (quarter) + c,c₁,-3.98776,1.04569,-3.814,0.0002,***
4,M10,Power (shared) + c,b₁,1.00409,0.49638,2.023,0.0437,*
5,M10,Power (shared) + c,c₁,-1.92686,1.18750,-1.623,0.1054,ns
6,M11,Power (free) + c,a₁,-0.03140,0.06167,-0.509,0.6110,ns
7,M11,Power (free) + c,b₁,2.85901,3.45469,0.828,0.4084,ns
8,M11,Power (free) + c,c₁,-4.68687,4.95827,-0.945,0.3451,ns
9,M12,Third-power + c,b₁,0.65961,0.25132,2.625,0.0090,**
